In [0]:
%run ./_utils_config

In [0]:
%run ./_utils_bronze_framework

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"Schema ready: {catalog}.{schema}")

In [0]:
print("Available source specs:", list_source_specs())

ree_spec = load_source_spec_file("ree")

errors = validate_source_spec(ree_spec)
print("✅ Spec valid" if not errors else f"❌ {errors}")

In [0]:
import copy


def broken_spec_errors(change):
    spec = copy.deepcopy(ree_spec)
    change(spec)
    return validate_source_spec(spec)


negative_tests = {
    "http instead of https": lambda s: s["endpoints"][0].update(url="http://apidatos.ree.es/es/datos"),
    "internal host": lambda s: s["endpoints"][0].update(url="https://localhost/admin"),
    "unknown placeholder": lambda s: s["endpoints"][0]["params"].update(token="{api_key}"),
    "POST method": lambda s: s["endpoints"][0].update(method="POST"),
    "days_back too large": lambda s: s["incremental"].update(days_back=5000),
    "missing {end}": lambda s: s["endpoints"][0]["params"].pop("end_date"),
    "query string in url": lambda s: s["endpoints"][0].update(url="https://apidatos.ree.es/x?a=1"),
    "unsupported source_type": lambda s: s.update(source_type="jdbc"),
    "missing source_type": lambda s: s.pop("source_type"),
    "csv_delimiter on json endpoint": lambda s: s["endpoints"][0].update(csv_delimiter=";"),
}

for test_name, change in negative_tests.items():
    errors = broken_spec_errors(change)
    print(f"{'✅ detected' if errors else '❌ NOT DETECTED'} · {test_name}: {errors}")

In [0]:
summary = run_source_spec(ree_spec, days_back_override=14)
print(f"\n{summary['chunks_ok']} ok / {summary['chunks_error']} errors · run_id {summary['run_id']}")
display(dbutils.fs.ls(f"{bronze_volume_path}/ree/prices"))

In [0]:
display(
    spark.table(ingestion_log_table)
    .orderBy(F.desc("ingested_at"), "endpoint_name", "chunk_start")
    .limit(100)
)

In [0]:
# Bronze tables test: run twice and check rows are replaced, not duplicated
def bronze_counts():
    counts = {}
    for ep in ree_spec["endpoints"]:
        table = bronze_table_name(ree_spec["source_name"], ep["name"])
        df = spark.table(table)
        counts[table] = (df.count(), df.select("source_file").distinct().count())
    return counts

run_source_spec(ree_spec, days_back_override=14)
counts_1 = bronze_counts()
run_source_spec(ree_spec, days_back_override=14)
counts_2 = bronze_counts()

print()
for table, (rows, files) in counts_2.items():
    print(f"{table}: {rows} rows / {files} files")
print(f"{'✅' if counts_1 == counts_2 else '❌'} Same row counts after re-running (no duplicates)")
print(f"{'✅' if all(r == f for r, f in counts_2.values()) else '❌'} One row per JSON file")

# Peek inside the VARIANT payload
display(
    spark.table(bronze_table_name("ree", "prices"))
    .select("chunk_start", "chunk_end", "source_file", "run_id",
            F.expr("payload:data.attributes.title::string").alias("dataset_title"),
            F.expr("payload:included[0].attributes.title::string").alias("first_series"))
    .orderBy("chunk_start")
)

In [0]:
display(
    spark.table(bronze_table_name("open_meteo", "hourly_weather"))
    .select("chunk_start", "chunk_end", "source_file",
            F.expr("payload:[0].latitude::double").alias("first_location_lat"),
            F.expr("payload:[1].latitude::double").alias("second_location_lat"),
            F.expr("payload:[0].hourly.time[0]::string").alias("first_hour"),
            F.expr("payload:[0].hourly.temperature_2m[0]::double").alias("first_temperature"))
    .orderBy("chunk_start")
)

In [0]:
reset_probe_budget()

result = probe_url(
    "https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real",
    {"start_date": "2026-11-10T00:00", "end_date": "2026-11-10T23:59", "time_trunc": "hour"},
)
print(json.dumps({k: v for k, v in result.items() if k != "sample"}, indent=2, ensure_ascii=False))

In [0]:
result = probe_url("https://date.nager.at/api/v3/PublicHolidays/2026/ES")
print(json.dumps({k: v for k, v in result.items() if k != "sample"}, indent=2, ensure_ascii=False))

In [0]:
safety_tests = {
    "http instead of https": ("http://date.nager.at/api/v3/PublicHolidays/2026/ES", None),
    "internal host": ("https://localhost/admin", None),
    "query string in url": ("https://date.nager.at/api/v3/PublicHolidays/2026/ES?x=1", None),
    "boolean param": ("https://date.nager.at/api/v3/PublicHolidays/2026/ES", {"flag": True}),
}

for test_name, (url, params) in safety_tests.items():
    result = probe_url(url, params)
    print(f"{'✅ rejected' if result['status'] == 'rejected' else '❌ NOT REJECTED'} · {test_name}: {result.get('errors')}")

In [0]:
result = probe_url("https://date.nager.at/api/v3/ThisDoesNotExist")
print(result["status"], result["http_status"], "· probes left:", result["probes_left"])

In [0]:
# a) Broken YAML: unquoted {start}
bad_yaml = """
source_name: test_bad_yaml
source_type: http_api
description: test
incremental:
  type: date_range
  days_back: 7
  chunk_days: 7
endpoints:
  - name: prices
    url: https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real
    response_format: json
    params:
      start_date: {start}T00:00
      end_date: {end}T23:59
"""
print("a)", json.loads(propose_source_spec(bad_yaml))["stage"], "\n")

# b) Valid YAML, but the source already exists
print("b)", json.loads(propose_source_spec(spec_to_yaml(load_source_spec_file("ree"))))["errors"], "\n")

# c) Valid spec, but the API path does not exist -> fails in the dry run
not_found_yaml = """
source_name: test_not_found
source_type: http_api
description: "Test: endpoint that does not exist"
incremental:
  type: snapshot
endpoints:
  - name: nothing
    url: https://date.nager.at/api/v3/ThisDoesNotExist
    method: GET
    response_format: json
"""
print("c)", json.loads(propose_source_spec(not_found_yaml))["stage"])

In [0]:
holidays_yaml = """
source_name: public_holidays_es
source_type: http_api
description: "Nager.Date public holidays API: national and regional public holidays in Spain for 2026."

auth:
  type: none

incremental:
  type: snapshot

endpoints:
  - name: holidays_2026
    url: https://date.nager.at/api/v3/PublicHolidays/2026/ES
    method: GET
    response_format: json
"""

print(propose_source_spec(holidays_yaml))

In [0]:
print("Source specs:", list_source_specs())

display(
    spark.table(bronze_table_name("public_holidays_es", "holidays_2026"))
    .select("source_file", "chunk_start", "run_id",
            F.expr("payload:[0].date::string").alias("first_date"),
            F.expr("payload:[0].localName::string").alias("first_holiday"),
            F.expr("array_size(cast(payload as array<variant>))").alias("holidays_in_file"))
)